In [ ]:
%matplotlib inline

# Deep Convolutional Generative Adversarial Network (DCGAN) Applied to CelebA Dataset and 128 x 128 images

Source:<P>

https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html

Adapted:<P>

Antonio Esteves @ UMinho, Jun 2024<P>

---

## Introduction

This tutorial will give an introduction to DCGANs through an example. We will train a generative adversarial network (GAN) to generate new celebrities after showing it pictures of many real celebrities. Most of the code here is from the DCGAN implementation in [pytorch/examples](https://github.com/pytorch/examples), and this document will give a thorough explanation of the implementation and shed light on how and why this model works. But don't worry, no prior knowledge of GANs is required, but it may require a first-timer to spend some time reasoning about what is actually happening under the hood. Also, for the sake of time it will help to have a GPU, or two. Lets start from the beginning.

## Generative Adversarial Networks


### What is a GAN?


GANs are a framework for teaching a deep learning model to capture the training data distribution so we can generate new data from that same distribution. GANs were invented by Ian Goodfellow in 2014 and first described in the paper [Generative Adversarial Nets](https://papers.nips.cc/paper/5423-generative-adversarial-nets.pdf). They are made of two distinct models, a *generator* and a *discriminator*. The job of the generator is to spawn 'fake' images that look like the training images. The job of the discriminator is to look at an image and output whether or not it is a real training image or a fake image from the generator. During training, the generator is constantly trying to outsmart the discriminator by generating better and better fakes, while the discriminator is working to become a better detective and correctly classify the real and fake images. The equilibrium of this game is when the generator is generating perfect fakes that look as if they came directly from the training data, and the discriminator is left to always guess at 50% confidence that the generator output is real or fake.

Now, lets define some notation to be used throughout tutorial starting with the discriminator. Let $x$ be data representing an image. $D(x)$ is the discriminator network which outputs the (scalar) probability that $x$ came from training data rather than the generator. Here, since we are dealing with images, the input to $D(x)$ is an image of CHW size 3x64x64. Intuitively, $D(x)$ should be HIGH when $x$ comes from training data and LOW when $x$ comes from the generator. $D(x)$ can also be thought of as a traditional binary classifier.

For the generator's notation, let $z$ be a latent space vector sampled from a standard normal distribution. $G(z)$ represents the generator function which maps the latent vector $z$ to data-space. The goal of $G$ is to estimate the distribution that the training data comes from ($p_{data}$) so it can generate fake samples from that estimated distribution ($p_g$).

So, $D(G(z))$ is the probability (scalar) that the output of the generator $G$ is a real image. As described in [Goodfellow's paper](https://papers.nips.cc/paper/5423-generative-adversarial-nets.pdf), $D$ and $G$ play a minimax game in which $D$ tries to maximize the probability it correctly classifies reals and fakes ($logD(x)$), and $G$ tries to minimize the probability that $D$ will predict its outputs are fake ($log(1-D(G(z)))$). From the paper, the GAN loss function is

$$\underset{G}{\text{min}} \underset{D}{\text{max}}V(D,G) = \mathbb{E}_{x\sim p_{data}(x)}\big[logD(x)\big] + \mathbb{E}_{z\sim p_{z}(z)}\big[log(1-D(G(z)))\big]$$

In theory, the solution to this minimax game is where $p_g = p_{data}$, and the discriminator guesses randomly if the inputs are real or fake. However, the convergence theory of GANs is still being actively researched and in reality models do not always train to this point.

### What is a DCGAN?


A DCGAN is a direct extension of the GAN described above, except that it explicitly uses convolutional and convolutional-transpose layers in the discriminator and generator, respectively. It was first described by Radford et. al. in the paper [Unsupervised Representation Learning With Deep Convolutional Generative Adversarial
Networks](https://arxiv.org/pdf/1511.06434.pdf). The discriminator is made up of strided
[convolution](https://pytorch.org/docs/stable/nn.html#torch.nn.Conv2d) layers, [batch
norm](https://pytorch.org/docs/stable/nn.html#torch.nn.BatchNorm2d) layers, and [LeakyReLU](https://pytorch.org/docs/stable/nn.html#torch.nn.LeakyReLU) activations. The input is a 3x64x64 input image and the output is a scalar probability that the input is from the real data distribution.
The generator is comprised of [convolutional-transpose](https://pytorch.org/docs/stable/nn.html#torch.nn.ConvTranspose2d) layers, batch norm layers, and [ReLU](https://pytorch.org/docs/stable/nn.html#relu) activations. The input is a latent vector, $z$, that is drawn from a standard normal distribution and the output is a 3x64x64 RGB image. The strided conv-transpose layers allow the latent vector to be transformed into a volume with the same shape as an image. In the paper, the authors also give some tips about how to setup the optimizers, how to calculate the loss functions, and how to initialize the model weights, all of which will be explained in the coming sections.

In [ ]:
import os
import random
import yaml
import wandb
import time

import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
import torchvision.datasets   as     dset
import torchvision.transforms as     transforms
import torchvision.utils      as     vutils

import numpy                  as     np
import matplotlib.pyplot      as     plt
import matplotlib.animation   as     animation
from   IPython.display        import HTML

## Set reproducible code

In [ ]:
# Set random seed for reproducibility
manualSeed = 999
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)
torch.use_deterministic_algorithms(True) # for reproducible results

## Hyperparameters

Let us define the hyperparameters for the run:

-   `dataset_path` - the path to the root of the dataset folder. We will
    talk more about the dataset in the next section.
-   `workers` - the number of worker threads for loading the data with
    the `DataLoader`.
-   `batch_size` - the batch size used in training. The DCGAN paper uses
    a batch size of 128.
-   `image_size` - the spatial size of the images used for training.
    This implementation defaults to 64x64. If another size is desired,
    the structures of D and G must be changed. See
    [here](https://github.com/pytorch/examples/issues/70) for more
    details.
-   `num_channels` - number of color channels in the input images. For color
    images this is 3.
-   `z_dim` - length of latent vector.
-   `g_channels` - relates to the depth of feature maps carried through the
    generator.
-   `d_channels` - sets the depth of feature maps propagated through the
    discriminator.
-   `num_epochs` - number of training epochs to run. Training for longer
    will probably lead to better results but will also take much longer.
-   `lr` - learning rate for training. As described in the DCGAN paper,
    this number should be 0.0002.
-   `beta1` - beta1 hyperparameter for Adam optimizers. As described in
    paper, this number should be 0.5.
-   `ngpu` - number of GPUs available. If this is 0, code will run in
    CPU mode. If this number is greater than 0 it will run on that
    number of GPUs.


### Read the configuration file

In [ ]:
LOAD_TRAINED_MODEL     = False
SKIP_TRAIN_MODEL       = False

CONFIG_FILE = '../config/dcgan_celeba_pttutorial_128x128_01.yaml'

with open(CONFIG_FILE, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

In [ ]:
print('parameters:')
for key, value in config.items():
    print(f'\t{key}: {value}')

In [ ]:
os.makedirs(f'results/{config["experiment_name"]}', exist_ok=True)

print(torch.__version__)

# Setup device agnostic code

device = "cuda:1" if torch.cuda.is_available() else "cpu"

print(f'Using {device} for computing')

## Login into Weights & Bias

In [ ]:
wandb.login()

## Track metadata and hyperparameters with Weights & Bias

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = config

wandb.init(
    project = 'OUR_WANDB_PROJECT_ID',
    entity  = 'OUR_WANDB_ENTITY', 
    config  = config_wandb
)

## Setup the data

In this notebook we will use the [Celeb-A Faces
dataset](http://mmlab.ie.cuhk.edu.hk/projects/CelebA.html) which can be
downloaded at the linked site, or in [Google
Drive](https://drive.google.com/drive/folders/0B7EVK8r0v71pTUZsaXdaSnZBZzg).
The dataset will download as a file named `img_align_celeba.zip`. Once downloaded, create a directory named `celeba` and extract the zip file into that directory. Then, set the `dataset_path` input for this notebook to the `celeba` directory you just created. The resulting directory structure should be:

``` {.sourceCode .sh}
/path/to/celeba
    -> img_align_celeba  
        -> 188242.jpg
        -> 173822.jpg
        -> 284702.jpg
        -> 537394.jpg
           ...
```

This is an important step because we will be using the `ImageFolder` dataset class, which requires there to be subdirectories in the dataset root folder. Now, we can create the dataset, create the dataloader, set the device to run on, and finally visualize some of the training data.

In [ ]:
# Create the dataset from the images in a folder

dataset = dset.ImageFolder(
    root=config["dataset_path"],
    transform=transforms.Compose([
        transforms.Resize(config['image_size']),
        transforms.CenterCrop(config['image_size']),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])
)

# Create the dataloader

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size  = config['batch_size'],
    shuffle     = True,
    num_workers = config['workers'],
)

# Plot some training images

real_batch = next(iter(dataloader))
plt.figure(figsize=(8,8))
plt.axis("off")
plt.title("Training Images")
plt.imshow(np.transpose(vutils.make_grid(real_batch[0].to(device)[:64], padding=2, normalize=True).cpu(),(1,2,0)))
plt.show()

## DCGAN implementation


With our input parameters set and the dataset prepared, we can now get into the implementation. We will start with the weight initialization strategy, then talk about the generator, discriminator, loss functions, and training loop in detail.

### Weight Initialization

From the DCGAN paper, the authors specify that all model weights shall be randomly initialized from a Normal distribution with `mean=0`, `stdev=0.02`. The `weights_init` function takes an initialized model as input and reinitializes all convolutional, convolutional-transpose, and batch normalization layers to meet this criteria. This function is applied to the models immediately after initialization.

In [ ]:
# custom weights initialization called on 'netG' and 'netD'

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

### Generator model

The generator, $G$, is designed to map the latent space vector ($z$) to data-space. Since our data are images, converting $z$ to data-space means ultimately creating a RGB image with the same size as the training images (i.e. 3x64x64). In practice, this is accomplished through a
series of strided two dime nsional convolutional transpose layers, each paired with a 2d batch norm layer and a relu activation. The output of the generator is fed through a tanh function to return it to the input data range of $[-1,1]$. It is worth noting the existence of the batch norm functions after the conv-transpose layers, as this is a critical contribution of the DCGAN paper. These layers help with the flow of gradients during training. An image of the generator from the DCGAN paper is shown below.

![](https://pytorch.org/tutorials/_static/img/dcgan_generator.png)

Notice, how the inputs we set in the input section (`config['z_dim']`, `config['g_channels']`, and `config['num_channels']`) influence the generator architecture in code. `config['z_dim']` is the length of the z input vector, `config['g_channels']` relates to the size of the feature maps
that are propagated through the generator, and `config['num_channels']` is the number of channels in the output image (set to 3 for RGB images). Below is the code for the generator.

In [ ]:
# Generator model

class Generator(nn.Module):
    def __init__(self, ngpu):
        super(Generator, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d( config['z_dim'], config['g_channels'] * 2, 4, 1, 0, bias=False),
            nn.BatchNorm2d(config['g_channels'] * 2),
            nn.ReLU(True),
            # state size: (config["g_channels"]*2 x 4 x 4) = (256 x 4 x 4)
            
            nn.ConvTranspose2d(config['g_channels'] * 2, config['g_channels'] * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(config['g_channels'] * 2),
            nn.ReLU(True),
            # state size: (config["g_channels"]*2 x 8 x 8) = (256 x 8 x 8)
            
            nn.ConvTranspose2d( config['g_channels'] * 2, config['g_channels'] * 1, 4, 2, 1, bias=False),
            nn.BatchNorm2d(config['g_channels'] * 1),
            nn.ReLU(True),
            # state size: (config["g_channels"]*1 x 16 x 16) = (128 x 16 x 16)
            
            nn.ConvTranspose2d( config['g_channels'] * 1, config['g_channels'] * 1, 4, 2, 1, bias=False),
            nn.BatchNorm2d(config['g_channels'] * 1),
            nn.ReLU(True),
            # state size: (config["g_channels"]*1 x 32 x 32) = ( 128 x 32 x 32)
            
            nn.ConvTranspose2d( config['g_channels'] * 1, int(config['g_channels']/2), 4, 2, 1, bias=False), # ADDED LAYER TO SUPPORT 128x128 IMAGES
            nn.BatchNorm2d(int(config['g_channels']/2)),
            nn.ReLU(True),
            # state size: (config["g_channels"]/2 x 64 x 64) = (64 x 64 x 64)

            nn.ConvTranspose2d( int(config['g_channels']/2), config['num_channels'], 4, 2, 1, bias=False),
            nn.Tanh()
            # state size: (config["num_channels"] x 128 x 128) = (3 x 128 x 128)
        )

    def forward(self, input):
        return self.main(input)

Now, we can instantiate the generator and apply the `weights_init` function. Check out the printed model to see how the generator object is structured.


In [ ]:
# Instantiate the generator

netG = Generator(config['ngpu']).to(device)

# Handle multi-GPU if desired

if (device == 'cuda') and (config['ngpu'] > 1):
    netG = nn.DataParallel(netG, list(range(config['ngpu'])))

# Apply the 'weights_init' function to randomly initialize all weights,
# in order they have 'mean=0' and stddev=0.02.

netG.apply(weights_init)

# Print the model architecture
print(netG)

### Discriminator mode

As mentioned, the discriminator, $D$, is a binary classification network that takes an image as input and outputs a scalar probability that the input image is real (as opposed to fake). Here, $D$ takes a 3x64x64 input image, processes it through a series of Conv2d, BatchNorm2d, and LeakyReLU layers, and outputs the final probability through a Sigmoid activation function. This architecture can be extended with more layers if necessary for the problem, but there is significance to the use of the strided convolution, BatchNorm, and LeakyReLUs. The DCGAN paper mentions it is a good practice to use strided convolution rather than pooling to downsample because it lets the network learn its own pooling function. Also batch norm and leaky relu functions promote healthy gradient flow which is critical for the learning process of both $G$ and $D$.

In [ ]:
# Discriminator model

class Discriminator(nn.Module):
    def __init__(self, ngpu):
        super(Discriminator, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # input shape: (config['num_channels'] x 128 x 128) = (3 x 128 x 128)

            nn.Conv2d(config['num_channels'], int(config['d_channels']/2), 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size: (config["d_channels"]/2 x 64 x 64) = (64 x 64 x 64)

            nn.Conv2d(int(config['d_channels']/2), config['d_channels'] * 1, 4, 2, 1, bias=False),
            nn.BatchNorm2d(config['d_channels'] * 1),
            nn.LeakyReLU(0.2, inplace=True),
            # state size: (config["d_channels"]*1 x 32 x 32) = (128 x 32 x 32)

            nn.Conv2d(config['d_channels'] * 1, config['d_channels'] * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(config['d_channels'] * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size: (config["d_channels"]*2 x 16 x 16) = (256 x 16 x 16)

            nn.Conv2d(config['d_channels'] * 2, config['d_channels'] * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(config['d_channels'] * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size: (config["d_channels"]*2 x 8 x 8) = (256 x 8 x 8)

            nn.Conv2d(config['d_channels'] * 2, config['d_channels'] * 4, 4, 2, 1, bias=False), # ADDED TO SUPPORT 128x128 IMAGES
            nn.BatchNorm2d(config['d_channels'] * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size: (config["d_channels"]*4 x 4 x 4) = (512 x 4 x 4)

            nn.Conv2d(config['d_channels'] * 4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input)

Now, as with the generator, we can create the discriminator, apply the `weights_init` function, and print the model's structure.


In [ ]:
# Instantiate the discriminator

netD = Discriminator(config['ngpu']).to(device)

# Handle multi-GPU if desired

if (device == 'cuda') and (config['ngpu'] > 1):
    netD = nn.DataParallel(netD, list(range(config['ngpu'])))

# Apply the 'weights_init' function to randomly initialize all weights,
# in order they have mean=0 and stddev=0.2.

netD.apply(weights_init)

# Print the model architecture
print(netD)

## Utility functions

In [ ]:
def time_format(seconds: int) -> str:
    if seconds is not None:
        seconds = int(seconds)
        d = seconds // (3600 * 24)
        h = seconds // 3600 % 24
        m = seconds % 3600 // 60
        s = seconds % 3600 % 60
        if d > 0:
            return '{:02d}D {:02d}H {:02d}m {:02d}s'.format(d, h, m, s)
        elif h > 0:
            return '{:02d}H {:02d}m {:02d}s'.format(h, m, s)
        elif m > 0:
            return '{:02d}m {:02d}s'.format(m, s)
        elif s > 0:
            return '{:02d}s'.format(s)
    return '-'


## Functions to save and load the models to/from file

In [ ]:
def save_model_and_results(discriminator, generator, results, hyperparameters, file_name):
    results_to_save = {
        'generator':       generator.state_dict(),
        'discriminator':   discriminator.state_dict(),
        'results':         results,
        'hyperparameters': hyperparameters,
    }

    torch.save(
        results_to_save,
        file_name,
    )

In [ ]:
def load_model(discriminator, generator, file_name, device):
    '''
    Given instances of the generator and discriminator models, loads from file 'file_name':
    (i)   the weights of both models,
    (ii)  the results obtained during model training and
    (iii) the training hyperparameters used to train the models,
    and put the models on 'device'.

    Returns the loaded results and the loaded hyperparameters.
    '''

    results_loaded = torch.load(file_name)

    generator.load_state_dict(results_loaded['generator'])
    generator.to(device)

    discriminator.load_state_dict(results_loaded['discriminator'])
    discriminator.to(device)

    # Returns the saved results and the saved hyperparameters
    return results_loaded['results'], results_loaded['hyperparameters']

## Loss functions and optimizers

With $D$ and $G$ setup, we can specify how they learn through the loss functions and optimizers. We will use the Binary Cross Entropy loss ([BCELoss](https://pytorch.org/docs/stable/generated/torch.nn.BCELoss.html#torch.nn.BCELoss)) function which is defined in PyTorch as:

$$\ell(x, y) = L = \{l_1,\dots,l_N\}^\top, \quad l_n = - \left[ y_n \cdot \log x_n + (1 - y_n) \cdot \log (1 - x_n) \right]$$

Notice how this function provides the calculation of both log components in the objective function (i.e. $log(D(x))$ and $log(1-D(G(z)))$). We can specify what part of the BCE equation to use with the $y$ input. This is accomplished in the training loop which is coming up soon, but it is important to understand how we can choose which component we wish to calculate just by changing $y$ (i.e. GT labels).

Next, we define our real label as 1 and the fake label as 0. These labels will be used when calculating the losses of $D$ and $G$, and this is also the convention used in the original GAN paper. Finally, we set up two separate optimizers, one for $D$ and one for $G$. As specified in the DCGAN paper, both are Adam optimizers with learning rate 0.0002 and Beta1 = 0.5. For keeping track of the generator's learning progression, we will generate a fixed batch of latent vectors that are drawn from a Gaussian distribution (i.e. fixed\_noise) . In the training loop, we will periodically input this fixed\_noise into $G$, and over the iterations we will see images form out of the noise.


In [ ]:
# Select the 'BCELoss' function

criterion = nn.BCELoss()

# Create batch of latent vectors that we will use to visualize
#  the progression of the generator

fixed_noise = torch.randn(64, config['z_dim'], 1, 1, device=device)

# Establish the convention for real and fake labels during training

real_label = 1.0
fake_label = 0.0

# Setup Adam optimizers for both generator and discriminator

optimizerD = optim.Adam(
    netD.parameters(),
    lr=config['lr'],
    betas=(config['beta1'], 0.999)
)

optimizerG = optim.Adam(
    netG.parameters(),
    lr=config['lr'],
    betas=(config['beta1'], 0.999)
)

## Training the DCGAN

Finally, now that we have all of the parts of the GAN framework defined, we can train it. Be mindful that training GANs is somewhat of an art form, as incorrect hyperparameter settings lead to mode collapse with little explanation of what went wrong. Here, we will closely follow Algorithm 1 from the [Goodfellow's paper](https://papers.nips.cc/paper/5423-generative-adversarial-nets.pdf), while abiding by some of the best practices shown in [ganhacks](https://github.com/soumith/ganhacks). Namely, we will "construct different mini-batches for real and fake" images, and also adjust G's objective function to maximize $log(D(G(z)))$. Training is split up into two main parts. Part 1 updates the Discriminator and Part 2 updates the Generator.

**Part 1 - Train the Discriminator**

Recall, the goal of training the discriminator is to maximize the probability of correctly classifying a given input as real or fake. In terms of Goodfellow, we wish to "update the discriminator by ascending its stochastic gradient". Practically, we want to maximize $log(D(x)) + log(1-D(G(z)))$. Due to the separate mini-batch suggestion from [ganhacks](https://github.com/soumith/ganhacks), we will calculate this in two steps. First, we will construct a batch of real samples from the training set, forward pass through $D$, calculate the loss ($log(D(x))$), then calculate the gradients in a backward pass. Secondly, we will construct a batch of fake samples with the current generator, forward pass this batch through $D$, calculate the loss ($log(1-D(G(z)))$), and *accumulate* the gradients with a backward pass. Now, with the gradients accumulated from both the all-real and all-fake batches, we call a step of the Discriminator's optimizer.

**Part 2 - Train the Generator**

As stated in the original paper, we want to train the Generator by minimizing $log(1-D(G(z)))$ in an effort to generate better fakes. As mentioned, this was shown by Goodfellow to not provide sufficient gradients, especially early in the learning process. As a fix, we instead wish to maximize $log(D(G(z)))$. In the code we accomplish this by: classifying the Generator output from Part 1 with the Discriminator, computing G's loss *using real labels as GT*, computing G's gradients in a backward pass, and finally updating G's parameters with an optimizer step. It may seem counter-intuitive to use the real labels as GT labels for the loss function, but this allows us to use the $log(x)$ part of the `BCELoss` (rather than the $log(1-x)$ part) which is exactly what we want.

Finally, we will do some statistic reporting and at the end of each epoch we will push our fixed\_noise batch through the generator to visually track the progress of G's training. The training statistics reported are:

-   **Loss\_D** - discriminator loss calculated as the sum of losses for the all real and all fake batches ($log(D(x)) + log(1 - D(G(z)))$).
-   **Loss\_G** - generator loss calculated as $log(D(G(z)))$ 
-   **D(x)** - the average output (across the batch) of the discriminator for the all real batch. This should start close to 1 then theoretically converge to 0.5 when G gets better. Think about why this is.
-   **D(G(z))** - average discriminator outputs for the all fake batch. The first number is before D is updated and the second number is after D is updated. These numbers should start near 0 and converge to 0.5 as G gets better. Think about why this is.

**Note:** This step might take a while, depending on how many epochs you run and if you removed some data from the dataset.


In [ ]:
def train_dcgan(discriminator, generator, optimizerD, optimizerG, loss_fn, dataloader, config, results, device):

    img_list = []
    iters    = 0

    print("Start training loop...")

    # For each epoch
    for epoch in range(config['num_epochs']):

        ts = time.time()  # start measuring time

        # For each batch in the dataloader
        for i, data in enumerate(dataloader, 0):

            #..................................................................
            # Update discriminator: maximize [log(D(x)) + log(1 - D(G(z)))]
            #..................................................................

            # Train D with a batch of real images .................

            discriminator.zero_grad() # Reset the gradients
            # Format the batch
            real_cpu  = data[0].to(device)
            b_size    = real_cpu.size(0)
            label     = torch.full((b_size,), real_label, dtype=torch.float, device=device)
            # Pass the batch of real images through D
            output    = discriminator(real_cpu).view(-1)
            # Calculate the loss on the batch of real images
            errD_real = loss_fn(output, label)
            # Calculate the gradient of the real images loss relative to D weights
            errD_real.backward()
            D_x       = output.mean().item()

            # Train D with a batch of fake images .................

            # Generate a batch of random vectors
            noise     = torch.randn(b_size, config['z_dim'], 1, 1, device=device)
            # Generate fake images from the random vectors using G
            fake      = generator(noise)
            label.fill_(fake_label)
            # Pass the batch of fake images through D
            output    = discriminator(fake.detach()).view(-1)
            # Calculate the loss on the batch of fake images
            errD_fake = loss_fn(output, label)
            # Calculate the gradient of the fake images loss relative to D weights
            errD_fake.backward()
            D_G_z1    = output.mean().item()
            # Compute D loss as a sum over the fake and the real batches
            errD      = errD_real + errD_fake
            # Update D using the gradients calculated from the losses
            optimizerD.step()

            #................................................................
            # Update generator: maximize log(D(G(z)))
            #................................................................

            generator.zero_grad() # Reset the gradients
            label.fill_(real_label)  # the generator loss is calciulated using real labels to trick D
            # Pass the same batch of fake images through D
            output = discriminator(fake).view(-1)
            # Calculate the generator loss
            errG = loss_fn(output, label)
            # Calculate the gradient of the G loss relative to G weights
            errG.backward()
            D_G_z2 = output.mean().item()
            # Update G using the gradients calculated from the losses
            optimizerG.step()

            # Save the results in a dictionary
            results["d_loss"].append(errD.item())
            results["g_loss"].append(errG.item())

            # Print progress metrics and save them to W&B .............................
            if i % config["log_interval"] == 0:

                mean_d_loss = np.mean(results["d_loss"][-config["log_interval"]:])
                mean_g_loss = np.mean(results["g_loss"][-config["log_interval"]:])

                print(f'epoch|iter: {epoch+1 :5d} | {i+1 :5d} / {len(dataloader) :6d} ({(i*100)/len(dataloader) :0>5.1f}%)', end="  ")
                print(f'D loss: {mean_d_loss :0>10.7f}', end="  ")
                print(f'G loss: {mean_g_loss :0>10.7f}', end="  ")
                print(f'D(x): {D_x :0>10.7f}  D(G(z1)): {D_G_z1 :0>10.7f}', end="  ")
                print(f'D(G(z2)): {D_G_z2 :0>10.7f}')

                try:
                    # Log metrics to Weights & Biases
                    wandb.log(
                        {
                        "discriminator_loss": mean_d_loss,
                        "generator_loss":     mean_g_loss,
                        "epoch":              epoch+1,
                        }
                    )
                except Exception as ex:
                    print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

                iters += 1

        # Check how the generator is doing by saving its output on 'fixed_noise'
        if (epoch % config["sampling_interval"] == 0) or (epoch == config['num_epochs']-1):

            generator.eval()
            with torch.inference_mode():
                fake = generator(fixed_noise).detach().cpu()
            generator.train()

            grid = vutils.make_grid(fake, padding=2, normalize=True)
            img_list.append(grid)
            grid = grid.permute(1, 2, 0)
            grid = grid.numpy()

            # Display the grid of images
            _ = plt.figure(figsize=(10, 10), constrained_layout=True)
            plt.imshow(grid)

            # Save the grid of images as a PNG file
            file_png = f'results/{config["experiment_name"]}/{config["experiment_name"]}_generated_epoch{str(epoch+1).zfill(3)}.png'
            plt.imsave(file_png, grid)

        # Log the epoch execution time in W&B
        te        = time.time()
        texec_sec = te - ts
        texec_str = time_format(texec_sec)
        print(f'Epoch training time: {texec_str}')

        try:
            wandb.log(
                {
                "epoch_training_time_sec": texec_sec,
                }
            )
        except Exception as ex:
            print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

        # Save the generator and discriminator in the 'models' directory
        if (epoch % config["checkp_interval"] == 0) or (epoch == config['num_epochs']-1):
            file_save_model = f'models/{config["experiment_name"]}.pth'
            save_model_and_results(
                discriminator,
                generator,
                results,
                config,
                file_save_model,
            )

    return img_list

In [ ]:
# Training Loop ...........................................................

# Create an empty dictionary to store the training results
results = {
    'd_loss': [],
    'g_loss': [],
    'epoch_training_time': 0.0,
}

# ==========================================================================================
# Train the model from the beginning
# ==========================================================================================

if LOAD_TRAINED_MODEL == False and SKIP_TRAIN_MODEL == False:

    img_list = train_dcgan(
        netD,
        netG,
        optimizerD,
        optimizerG,
        criterion,
        dataloader,
        config,
        results,
        device,
    )

# ==========================================================================================
# Load the saved models
# ==========================================================================================

elif LOAD_TRAINED_MODEL == True:

    file_save_model = f'models/{config["experiment_name"]}.pth'
    results, _      = load_model(
        netD,
        netG,
        file_save_model,
        device,
    )

    # --------------------------------------------------------------------------------------
    # Continue training of the loaded models
    # --------------------------------------------------------------------------------------

    if SKIP_TRAIN_MODEL == False:

        img_list = train_dcgan(
            netD,
            netG,
            optimizerD,
            optimizerG,
            criterion,
            dataloader,
            config,
            results,
            device,
        )

## Results

Finally, lets check out how we did. Here, we will look at three different results. First, we will see how D and G's losses changed during training. Second, we will visualize G's output on the fixed\_noise batch for every epoch. And third, we will look at a batch of real data next to a batch of fake data from G.

### Plot the loss curves

Below is a plot of discriminator and generator losses as function of the training iterations.

In [ ]:
if SKIP_TRAIN_MODEL == False:

    plt.figure(figsize=(10,5))
    plt.title("DCGAN generator and discriminator losses during training")
    plt.plot(results["g_loss"],label="G loss")
    plt.plot(results["d_loss"],label="D loss")
    plt.xlabel("iteration")
    plt.ylabel("loss")
    plt.legend()
    plt.savefig(f'results/{config["experiment_name"]}/{config["experiment_name"]}_losses.png')
    plt.show()

### Visualize the Generator optimization

Remember how we saved the generator output on the fixed noise batch after every epoch of training. Now, we can visualize the training progression of the generator with an animation. Press the play button to start the animation.


In [ ]:
if SKIP_TRAIN_MODEL == False:
    fig = plt.figure(figsize=(8,8))
    plt.axis("off")
    ims = [[plt.imshow(np.transpose(i,(1,2,0)), animated=True)] for i in img_list]
    ani = animation.ArtistAnimation(fig, ims, interval=1000, repeat_delay=1000, blit=True)

    HTML(ani.to_jshtml())

**Real Images vs. Fake Images**

Finally, lets take a look at some real images and fake images side by
side.


In [ ]:
if SKIP_TRAIN_MODEL == False:

    # Grab a batch of real images from the dataloader
    real_batch = next(iter(dataloader))

    # Plot the real images
    plt.figure(figsize=(15,15))
    plt.subplot(1,2,1)
    plt.axis("off")
    plt.title("Real Images")
    plt.imshow(np.transpose(vutils.make_grid(real_batch[0].to(device)[:64], padding=5, normalize=True).cpu(),(1,2,0)))

    # Plot the fake images from the last epoch
    plt.subplot(1,2,2)
    plt.axis("off")
    plt.title("Fake Images")
    plt.imshow(np.transpose(img_list[-1],(1,2,0)))
    plt.savefig(f'results/{config["experiment_name"]}/{config["experiment_name"]}_real_vs_generated.png')
    plt.show()


## Generate grids of images with fully trained generator

In [ ]:
def generate_grid_images(generator, num_grids, grid_size, config, device):

    generator.eval()

    with torch.inference_mode():

        for num in range(num_grids):

            # Generate a set of latent vectors
            noise = torch.randn(grid_size, config['z_dim'], 1, 1, device=device)

            # Generate a set of fake images with G
            fake = generator(noise).detach().cpu()

            # Create a grid with the generated images
            grid = vutils.make_grid(fake, padding=2, normalize=True)
            grid = grid.permute(1, 2, 0)
            grid = grid.numpy()

            # Display the grid of images
            fig = plt.figure(figsize=(10, 10), constrained_layout=True)
            plt.imshow(grid)

            # Save the grid of images as a PNG file
            file_png = f'results/{config["experiment_name"]}/{config["experiment_name"]}_generated_final_{str(num+1).zfill(3)}.png'
            plt.imsave(file_png, grid)

In [ ]:
generate_grid_images(netG, 8, 64, config, device)

Where to Go Next
================

We have reached the end of our journey, but there are several places you
could go from here. You could:

-   Train for longer to see how good the results get
-   Modify this model to take a different dataset and possibly change
    the size of the images and the model architecture
-   Check out some other cool GAN projects
    [here](https://github.com/nashory/gans-awesome-applications)
-   Create GANs that generate
    [music](https://www.deepmind.com/blog/wavenet-a-generative-model-for-raw-audio/)


In [ ]:
# Mark the Weights & Bias run as finished
wandb.finish()